In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

# Data
data = {
    'Year': ['2021-Q1', '2021-Q2', '2021-Q3', '2021-Q4', '2022-Q1', '2022-Q2', 
             '2022-Q3', '2022-Q4', '2023-Q1', '2023-Q2', '2023-Q3', '2023-Q4', 
             '2024-Q1', '2024-Q2', '2024-Q3', '2024-Q4'],
    'Investments': [12, 24, 38, 32, 56, 51, 71, 89, 116, 171, 180, 149, 143, 154, 168, 200]
}

# Create DataFrame
df = pd.DataFrame(data)
df['Year'] = pd.to_datetime(df['Year'].str.replace('Q', '') + '-1', format='%Y-%q-%w')
df.set_index('Year', inplace=True)

# Check stationarity
result = adfuller(df['Investments'])
print('ADF Statistic:', result[0])
print('p-value:', result[1])

# Fit SARIMA model
model = SARIMAX(df['Investments'], order=(1, 1, 1), seasonal_order=(1, 1, 1, 4))
results = model.fit()

# Forecasting for the next 8 quarters
forecast = results.get_forecast(steps=8)
forecast_index = pd.date_range(start='2025-01-01', periods=8, freq='Q')
forecast_values = forecast.predicted_mean

# Plotting
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['Investments'], label='Historical Investments', marker='o')
plt.plot(forecast_index, forecast_values, label='Forecast', color='red', marker='o')
plt.fill_between(forecast_index, 
                 forecast.conf_int()['lower Investments'], 
                 forecast.conf_int()['upper Investments'], 
                 color='pink', alpha=0.3)
plt.title('Investment Forecast using SARIMA')
plt.xlabel('Year')
plt.ylabel('Investments')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

ValueError: 'q' is a bad directive in format '%Y-%q-%w'